In [1]:
import sys
print(sys.executable)

c:\ProgramData\anaconda3\envs\museum_chroma\python.exe


In [1]:
import json
from pathlib import Path
from tqdm import tqdm

import chromadb
from chromadb.utils import embedding_functions


# =========================
# 1. 경로 설정
# =========================

JSON_PATH = "museum_collections.json"
DB_DIR = "chroma_museum_db"
COLLECTION_NAME = "museum_relics"


# =========================
# 2. JSON 로드
# =========================

json_path = Path(JSON_PATH)

if not json_path.exists():
    raise FileNotFoundError(f"JSON 파일을 찾을 수 없습니다: {JSON_PATH}")

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"로드된 유물 수: {len(data)}")


# =========================
# 3. 사용할 필드만 추출
# =========================

documents = []
metadatas = []
ids = []

for idx, item in enumerate(data):
    relic_id = item.get("relicId", idx)
    title = item.get("title", "")
    other_name = item.get("다른명칭", "")
    exhibition_name = item.get("전시명칭", "")
    collection_no = item.get("소장품번호", "")
    description = item.get("description", "")

    # 검색에 사용될 텍스트
    text = f"""
유물ID: {relic_id}
유물명: {title}
다른명칭: {other_name}
전시명칭: {exhibition_name}
소장품번호: {collection_no}
설명문: {description}
""".strip()

    documents.append(text)

    # 검색 결과로 확인할 metadata
    metadatas.append({
        "relicId": int(relic_id) if str(relic_id).isdigit() else str(relic_id),
        "title": str(title),
        "다른명칭": str(other_name),
        "전시명칭": str(exhibition_name),
        "소장품번호": str(collection_no),
        "description": str(description)
    })

    # Chroma 내부 고유 ID
    ids.append(f"relic_{relic_id}")

print(f"DB에 넣을 유물 수: {len(documents)}")


# =========================
# 4. 임베딩 모델 설정
# =========================

embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="jhgan/ko-sroberta-multitask"
)


# =========================
# 5. Chroma DB 연결
# =========================

client = chromadb.PersistentClient(path=DB_DIR)


# =========================
# 6. 기존 컬렉션 삭제
# =========================

try:
    client.delete_collection(name=COLLECTION_NAME)
    print(f"기존 컬렉션 삭제 완료: {COLLECTION_NAME}")
except Exception as e:
    print(f"기존 컬렉션 없음 또는 삭제 생략: {e}")


# =========================
# 7. 새 컬렉션 생성
# =========================

collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function,
    metadata={
        "description": "Museum relic DB using relicId, title, 다른명칭, 전시명칭, 소장품번호, description"
    }
)


# =========================
# 8. 배치 저장
# =========================

BATCH_SIZE = 100

for start in tqdm(range(0, len(documents), BATCH_SIZE), desc="Chroma DB 재구축 중"):
    end = start + BATCH_SIZE

    collection.add(
        ids=ids[start:end],
        documents=documents[start:end],
        metadatas=metadatas[start:end]
    )

print("Chroma DB 재구축 완료")
print(f"저장 위치: {DB_DIR}")
print(f"컬렉션 이름: {COLLECTION_NAME}")
print(f"총 저장 개수: {collection.count()}")

로드된 유물 수: 2560
DB에 넣을 유물 수: 2560


c:\ProgramData\anaconda3\envs\museum_chroma\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7960.12it/s]


기존 컬렉션 없음 또는 삭제 생략: Collection [museum_relics] does not exist


Chroma DB 재구축 중: 100%|██████████| 26/26 [02:20<00:00,  5.39s/it]

Chroma DB 재구축 완료
저장 위치: chroma_museum_db
컬렉션 이름: museum_relics
총 저장 개수: 2560


In [2]:
import chromadb
from chromadb.utils import embedding_functions

DB_DIR = "chroma_museum_db"
COLLECTION_NAME = "museum_relics"

embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="jhgan/ko-sroberta-multitask"
)

client = chromadb.PersistentClient(path=DB_DIR)

collection = client.get_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function
)

query = "청자 모란무늬 항아리"

results = collection.query(
    query_texts=[query],
    n_results=5
)

for i in range(len(results["ids"][0])):
    meta = results["metadatas"][0][i]

    print("=" * 80)
    print("순위:", i + 1)
    print("거리:", results["distances"][0][i])
    print("Chroma ID:", results["ids"][0][i])
    print("relicId:", meta["relicId"])
    print("유물명:", meta["title"])
    print("다른명칭:", meta["다른명칭"])
    print("전시명칭:", meta["전시명칭"])
    print("소장품번호:", meta["소장품번호"])
    print("설명:", meta["description"][:300])

순위: 1
거리: 0.22575759887695312
Chroma ID: relic_1473
relicId: 1473
유물명: 청자 양각 모란무늬 항아리
다른명칭: 靑磁陽刻牡丹文壺
전시명칭: 
소장품번호: 덕수6207
설명: 이 항아리는 작고 말린 입에 납작하면서 둥근 복부에 굽은 권족(圈足)의 형태이다. 녹청색 유약이 두껍게 발라져 있고 기포가 양각된 파인 부분에 모여 있다. 외벽에 큼직한 한 송이의 모란 꽃잎이 몸통 전체에 양각되어 있어 마치 모란 한 송이 같은 느낌을 준다. 전체적으로 유약이 입혀졌고 바닥의 접지면에 불에 강한 내화토를 받친 흔적이 있다.
순위: 2
거리: 0.2526528835296631
Chroma ID: relic_1546
relicId: 1546
유물명: 청자 연꽃 국화 모란무늬 두 귀 달린 항아리
다른명칭: 靑磁陽刻蓮菊牧丹文兩耳壺, 청자 양각 연국모란문 양이 호
전시명칭: 
소장품번호: 신안1004
설명: 짧은 목에 어깨가 벌어져 있는 항아리로 어깨 양쪽에 동그란 귀가 달려 있다. 몸통 중앙부에 선을 돌려 구분하고 위에는 연꽃, 국화, 모란 무늬를, 아래는 넝쿨 무늬를 양각(陽刻)하였다.
순위: 3
거리: 0.25444352626800537
Chroma ID: relic_1559
relicId: 1559
유물명: 청자 꽃무늬 항아리
다른명칭: 靑磁陰刻雷文壺, 청자 음각 뇌문 호, 청자 음각 번개무늬 항아리
전시명칭: 청자 작은 단지
소장품번호: 신안1891
설명: 신안1891번은 항아리, 신안1892번은 뚜껑이다. 몸통의 상하에 음각(陰刻) 선을 두르고 그 안쪽에 번개 무늬(雷文)를 넣었다. 몸통 아랫부분에는 연꽃잎 무늬(蓮瓣文)를 음각(陰刻)하여 돌렸으며, 뚜껑에도 연꽃잎 무늬를 가늘게 음각하였다. 항아리의 바닥은 구멍이 뚫린 채로 두고 안쪽에서 둥근 형태의 바닥을 따로 만들어 붙였다. 광택이 나는 짙은 녹색의 유약을 사용하였다. 유약을 바르지 않은 항아리의 접지면과 구연부, 뚜껑의 안쪽은 붉은색의 태토(胎土)가 드러나 있다.


In [3]:
import chromadb
import sentence_transformers
import transformers
import torch
import sys

print("python:", sys.version)
print("chromadb:", chromadb.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)

python: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
chromadb: 1.5.9
sentence-transformers: 5.5.1
transformers: 5.8.1
torch: 2.12.0+cpu
